# US-China Tension Index - Exploratory Data Analysis

This notebook explores the US-China Tension (UCT) Index dataset.

**Dataset:** `us-china-tension.csv`

**Description:** Monthly US-China Tension Index from 1993 to 2024.

**Citation:** Rogers, Sun, and Sun (2024), U.S.-China Tension, Working paper

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

DATA_PATH = Path('../../datasets/raw/us-china-tension.csv')

## 1. Load and Clean Data

In [ ]:
# Load dataset
df = pd.read_csv(DATA_PATH)

print("First 5 rows (raw):")
display(df.head())

# Keep only the first two columns (date and UCT)
df = df.iloc[:, :2]
df.columns = ['date', 'UCT']

# Remove rows with missing UCT values
df = df.dropna(subset=['UCT'])

# Parse date (format: YYYYmM)
df['date'] = pd.to_datetime(df['date'], format='%Ym%m')

# Extract year and month
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month

# Sort by date
df = df.sort_values('date').reset_index(drop=True)

print("\nCleaned dataset:")
display(df.head())
print("\nData types:")
print(df.dtypes)

## 2. Data Overview

In [ ]:
print("Dataset Shape:", df.shape)
print("\nDate range:", df['date'].min(), "to", df['date'].max())
print("Total months:", len(df))
print("\nDataset Info:")
df.info()

## 3. Descriptive Statistics

In [ ]:
print("Descriptive Statistics for UCT Index:")
display(df['UCT'].describe())

print("\nAdditional Statistics:")
print(f"Skewness: {df['UCT'].skew():.4f}")
print(f"Kurtosis: {df['UCT'].kurtosis():.4f}")

## 4. Time Series Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(df['date'], df['UCT'], linewidth=1.5, marker='o', markersize=2, alpha=0.8)
ax.set_title('US-China Tension Index (1993-2024)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('UCT Index', fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# With moving average
df['MA_12m'] = df['UCT'].rolling(window=12, center=True).mean()

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(df['date'], df['UCT'], linewidth=1, alpha=0.5, label='Monthly UCT')
ax.plot(df['date'], df['MA_12m'], linewidth=2.5, label='12-month MA', color='red')
ax.set_title('US-China Tension Index with 12-Month Moving Average', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('UCT Index', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['UCT'], bins=30, edgecolor='black', alpha=0.7)
axes[0].axvline(df['UCT'].mean(), color='red', linestyle='--', label=f'Mean: {df["UCT"].mean():.2f}')
axes[0].axvline(df['UCT'].median(), color='green', linestyle='--', label=f'Median: {df["UCT"].median():.2f}')
axes[0].set_title('Distribution of UCT Index', fontsize=12, fontweight='bold')
axes[0].set_xlabel('UCT Index')
axes[0].set_ylabel('Frequency')
axes[0].legend()

axes[1].boxplot(df['UCT'], vert=True)
axes[1].set_title('Box Plot of UCT Index', fontsize=12, fontweight='bold')
axes[1].set_ylabel('UCT Index')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Temporal Patterns

In [ ]:
# Yearly statistics
yearly_stats = df.groupby('year')['UCT'].agg(['mean', 'std', 'min', 'max', 'count'])
print("Yearly Statistics:")
display(yearly_stats)

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(yearly_stats.index, yearly_stats['mean'], alpha=0.7, edgecolor='black')
ax.set_title('Average UCT Index by Year', fontsize=14, fontweight='bold')
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Average UCT Index', fontsize=12)
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 7. Key Events Analysis

In [ ]:
# Identify peaks (highest tension periods)
top_10_tension = df.nlargest(10, 'UCT')[['date', 'UCT']]
print("Top 10 Highest Tension Periods:")
display(top_10_tension)

# Identify lowest tension periods
bottom_10_tension = df.nsmallest(10, 'UCT')[['date', 'UCT']]
print("\nTop 10 Lowest Tension Periods:")
display(bottom_10_tension)

## 8. Change Analysis

In [ ]:
# Calculate month-over-month change
df['UCT_change'] = df['UCT'].diff()
df['UCT_pct_change'] = df['UCT'].pct_change() * 100

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

axes[0].bar(df['date'], df['UCT_change'], alpha=0.7, width=20)
axes[0].axhline(y=0, color='black', linestyle='--', linewidth=1)
axes[0].set_title('Month-over-Month Change in UCT Index', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Change in UCT')
axes[0].grid(True, alpha=0.3)

axes[1].hist(df['UCT_change'].dropna(), bins=30, edgecolor='black', alpha=0.7)
axes[1].axvline(df['UCT_change'].mean(), color='red', linestyle='--', label=f'Mean: {df["UCT_change"].mean():.2f}')
axes[1].set_title('Distribution of Monthly Changes', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Change in UCT')
axes[1].set_ylabel('Frequency')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Summary

In [ ]:
print("=" * 60)
print("KEY FINDINGS - US-CHINA TENSION INDEX")
print("=" * 60)
print(f"\n1. Dataset Coverage:")
print(f"   - Start: {df['date'].min().strftime('%Y-%m')}")
print(f"   - End: {df['date'].max().strftime('%Y-%m')}")
print(f"   - Total Months: {len(df)}")
print(f"\n2. UCT Index Statistics:")
print(f"   - Mean: {df['UCT'].mean():.2f}")
print(f"   - Median: {df['UCT'].median():.2f}")
print(f"   - Std Dev: {df['UCT'].std():.2f}")
print(f"   - Min: {df['UCT'].min():.2f} ({df.loc[df['UCT'].idxmin(), 'date'].strftime('%Y-%m')})")
print(f"   - Max: {df['UCT'].max():.2f} ({df.loc[df['UCT'].idxmax(), 'date'].strftime('%Y-%m')})")
print(f"\n3. Temporal Insights:")
print(f"   - Year with highest avg: {yearly_stats['mean'].idxmax()} ({yearly_stats['mean'].max():.2f})")
print(f"   - Year with lowest avg: {yearly_stats['mean'].idxmin()} ({yearly_stats['mean'].min():.2f})")
print(f"\n4. Volatility:")
print(f"   - Avg Monthly Change: {df['UCT_change'].mean():.2f}")
print(f"   - Std of Changes: {df['UCT_change'].std():.2f}")
print("\n" + "=" * 60)